# Company Report Visualization

### Setup
Load the analysis workbook and define the chart helpers.

In [1]:
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

import sector_mapping

XLSX = os.path.join(sector_mapping.REPORTS_DIR, "complete_company_analysis.xlsx")
history = pd.read_excel(XLSX, sheet_name="1_Historical_All_Quarters", parse_dates=["FiscalDateEnding"])
latest = pd.read_excel(XLSX, sheet_name="2_Latest_Quarter_Complete").dropna(subset=["CurrentPrice", "FairValue_Composite"])

PALETTE = ["#ef4444", "#f97316", "#eab308", "#22c55e", "#06b6d4", "#3b82f6", "#8b5cf6", "#ec4899"]
AXIS = dict(showgrid=True, gridcolor="#e5e7eb", zeroline=False, showline=True, linecolor="#d1d5db", tickfont=dict(size=11))
CLIP_COLS = ["TotalRevenue", "NetIncome", "GrossMargin", "RevenueGrowth_YoY", "TotalAssets", "TotalLiabilities", "TotalShareholderEquity"]


def layout(fig, title, x_title=None, y_title=None, height=420, **extra):
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color="#111827"), x=0.02, xanchor="left"),
        template="plotly_white", height=height, colorway=PALETTE,
        font=dict(family="Inter, system-ui, -apple-system, sans-serif", size=12, color="#374151"),
        paper_bgcolor="#ffffff", plot_bgcolor="#fafafa", margin=dict(t=60, b=60, l=60, r=40),
        xaxis=dict(title=x_title, **AXIS), yaxis=dict(title=y_title, **AXIS), hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                    bgcolor="rgba(255,255,255,0.9)", bordercolor="#e5e7eb", borderwidth=1),
        **extra,
    )
    display(HTML(fig.to_html(include_plotlyjs="cdn")))


def line_chart(d, traces, title, yaxis_title, hline=False):
    """traces: [(column, label, color)]."""
    fig = go.Figure()
    for col, name, color in traces:
        fig.add_trace(go.Scatter(x=d["FiscalDateEnding"], y=d[col], name=name, mode="lines+markers",
                                 line=dict(color=color, width=2.5), marker=dict(size=6)))
    if hline:
        fig.add_hline(y=0, line_dash="dash", line_color="#9ca3af", line_width=1)
    layout(fig, title, "Fiscal Date Ending", yaxis_title)


def bar_chart(d, traces, title, value_title, x="Symbol", horizontal=False):
    fig = go.Figure()
    for col, name, color in traces:
        vals = d[col].round(2)
        fig.add_trace(go.Bar(**{("y" if horizontal else "x"): d[x], ("x" if horizontal else "y"): vals},
                             orientation="h" if horizontal else "v", name=name, text=vals, textposition="outside",
                             marker=dict(color=color, line=dict(width=0)), textfont=dict(size=11)))
    if horizontal:
        layout(fig, title, x_title=value_title, height=max(420, len(d) * 28), barmode="group")
    else:
        layout(fig, title, y_title=value_title, barmode="group")

## Single Stock

In [2]:
# Pick the stock via env var (e.g. `COMPANY_SYMBOL=NVDA jupyter nbconvert ...`) or edit the default here.
SYMBOL = os.getenv("COMPANY_SYMBOL", "UMAC").strip().upper()
available = sorted(history["Symbol"].unique())
if SYMBOL not in available:
    print(f"{SYMBOL} not in complete_company_analysis.xlsx - showing {available[0]} instead. Available: {', '.join(available)}")
    SYMBOL = available[0]

d = history[history["Symbol"] == SYMBOL].sort_values("FiscalDateEnding").copy()
for col in CLIP_COLS:
    if d[col].notna().sum() > 2:
        d[col] = d[col].clip(*d[col].quantile([0.02, 0.98]))

line_chart(d, [("TotalRevenue", "Revenue", "#ef4444"), ("NetIncome", "Net Income", "#22c55e")], f"{SYMBOL} – Revenue & Net Income", "$ Millions")
line_chart(d, [("RevenueGrowth_YoY", "Revenue Growth YoY", "#3b82f6")], f"{SYMBOL} – Revenue Growth YoY (%)", "%", hline=True)
line_chart(d, [("GrossMargin", "Gross Margin", "#8b5cf6")], f"{SYMBOL} – Gross Margin (%)", "%")
line_chart(d, [("TotalAssets", "Total Assets", "#ef4444"), ("TotalLiabilities", "Total Liabilities", "#f97316"),
               ("TotalShareholderEquity", "Equity", "#22c55e")], f"{SYMBOL} – Balance Sheet ($M)", "$ Millions")


def trimmed_mean(s):
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3:
        return s.mean()
    low, high = s.quantile([0.02, 0.98])
    return s[(s >= low) & (s <= high)].mean()


sector_avg = latest.groupby("Sector").agg(SectorAvg_PE=("PE_Ratio", trimmed_mean), SectorAvg_PB=("PB_Ratio", trimmed_mean))
row = latest[latest["Symbol"] == SYMBOL].merge(sector_avg, left_on="Sector", right_index=True)
if row.empty:
    print(f"No price / fair value for {SYMBOL}")
else:
    sector = row["Sector"].iloc[0]
    bar_chart(row, [("CurrentPrice", "Current Price", "#06b6d4"), ("FairValue_Composite", "Fair Value", "#3b82f6")],
              f"{SYMBOL} – Fair Value vs Current Price | Sector: {sector}", "$")
    bar_chart(row, [("PE_Ratio", "P/E", "#ef4444"), ("PB_Ratio", "P/B", "#22c55e"),
                    ("SectorAvg_PE", "Sector Avg P/E", "#8b5cf6"), ("SectorAvg_PB", "Sector Avg P/B", "#ec4899")],
              f"{SYMBOL} – P/E, P/B vs Sector Averages | Sector: {sector}", "Ratio")

## Fair Value vs Price by Sector

In [3]:
for sector, group in latest.sort_values("Symbol").groupby("Sector"):
    bar_chart(group, [("CurrentPrice", "Current Price", "#06b6d4"), ("FairValue_Composite", "Fair Value", "#3b82f6")],
              f"{sector} – Fair Value vs Current Price", "$", horizontal=True)